In [2]:
# Cell 1 – Simulasi utama (fixed)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NPM = 19102
np.random.seed(NPM)

makan = pd.read_csv("makanan.csv")

budget0 = 1e6  # initial budget

wkenyang_base = 0.8
wrasa_base    = 0.7
wsehat_base   = 0.9
wbosan_base   = 0.2

SAMPLE_NUM = 500
NUM_DAYS = 31
sampleid = np.arange(SAMPLE_NUM)

skor = np.zeros(SAMPLE_NUM, dtype=float)
sisa = np.zeros(SAMPLE_NUM, dtype=float)
sehat = np.zeros(SAMPLE_NUM, dtype=float)

SISA_MININUM = 150e3
SISA_MAKSIMUM = 300e3
bin_sisa = np.arange(SISA_MININUM, SISA_MAKSIMUM, 1000)
resp_sehat = np.zeros((SAMPLE_NUM, len(bin_sisa)), dtype=float)

# FIX: harus NUM_DAYS * 2 kolom (2 makan per hari)
resp_pilih = np.full((SAMPLE_NUM, NUM_DAYS * 2), -1, dtype=int)

for n in range(SAMPLE_NUM):
    budget = budget0
    skor_kumulatif = 0.0
    skor_sehat = 0.0

    for i in range(NUM_DAYS):
        # makan pertama (pagi)
        pilihan = np.random.randint(0, len(makan))
        w_kenyang = np.random.normal(loc=wkenyang_base, scale=0.2)
        w_rasa    = np.random.normal(loc=wrasa_base, scale=0.2)
        w_sehat   = np.random.normal(loc=wsehat_base, scale=0.4)
        w_bosan   = np.random.normal(loc=wbosan_base, scale=0.2)

        skor_kumulatif += (
            w_kenyang * makan.loc[pilihan, 'kenyang'] +
            w_rasa    * makan.loc[pilihan, 'rasa'] +
            w_sehat   * makan.loc[pilihan, 'sehat'] -
            w_bosan   * makan.loc[pilihan, 'bosan']
        )
        skor_sehat += makan.loc[pilihan, 'sehat'] - 0.5
        budget -= makan.loc[pilihan, 'harga']
        resp_pilih[n, 2 * i] = pilihan

        # makan kedua (malam)
        pilihan = np.random.randint(0, len(makan))
        w_kenyang = np.random.normal(loc=wkenyang_base, scale=0.2)
        w_rasa    = np.random.normal(loc=wrasa_base, scale=0.2)
        w_sehat   = np.random.normal(loc=wsehat_base + 0.2, scale=0.4)
        w_bosan   = np.random.normal(loc=wbosan_base * 2, scale=0.2)

        skor_kumulatif += (
            w_kenyang * makan.loc[pilihan, 'kenyang'] +
            w_rasa    * makan.loc[pilihan, 'rasa'] +
            w_sehat   * makan.loc[pilihan, 'sehat'] -
            w_bosan   * makan.loc[pilihan, 'bosan']
        )
        skor_sehat += makan.loc[pilihan, 'sehat'] - 0.5
        budget -= makan.loc[pilihan, 'harga']
        resp_pilih[n, 2 * i + 1] = pilihan

    sisa[n] = budget
    skor[n] = skor_kumulatif
    sehat[n] = skor_sehat

    if SISA_MININUM <= budget < SISA_MAKSIMUM:
        idx_bin = int((budget - SISA_MININUM) / 1000)
        resp_sehat[n, idx_bin] = skor_sehat

# quick shape checks
print("resp_pilih.shape =", resp_pilih.shape)   # harus (500, 62)
print("resp_sehat.shape =", resp_sehat.shape)


resp_pilih.shape = (500, 62)
resp_sehat.shape = (500, 150)


Kode diatas merupakan kode untuk membuat program simulasi selama 31 hari atau sebulan penuh dengan asumsi 2 kali makan per hari yaitu pagi dan malam. Dataset makanan diatas berisi variabel kenyang, rasa, sehat, bosan dan harga mengenai makanannya. 

Pada output tertulis bahwa resp_pilih.shape terdapat 500 baris yang dimana setiap baris menyimpan pilihan makanan untuk 31 hari dan dikalikan dengan 2 makan perhari yakni 62. Jadi tabel ini menyimpan makanan apa saja yang dipilih setiap hari dan dimakan waktu makan. Begitupun dengan tabel reso_sehat terdapat 500 baris juga dengan setiap kolom merupakan jumlah bin sisa uang dari yang Rp. 1500000 sampai Rp. 300000. Setiap entri menyimpan skor kesehatan kumulatif jika sisa uang jatuh pada bin tertentu, karena sebagian kecil baris yang sisa uangnya masuk dalam rentang ini, maka sebagian besar elemen akan bernilai 0. 

In [5]:
# (1) cari pola makan mana saja yang menghasilkan nilai kumulatif kesehatan (resp_sehat) positif
idx_sehat_pos = np.where(sehat > 0)[0]

print("Jumlah pola makan dengan kesehatan positif:", len(idx_sehat_pos))
print("Contoh indeks simulasi sehat positif:", idx_sehat_pos[:10])


Jumlah pola makan dengan kesehatan positif: 119
Contoh indeks simulasi sehat positif: [ 0  1  2 12 14 20 21 30 31 39]


Pada source code diatas digunakan untuk mencari semua indeks simulasi yang memiliki skor sehat lebih dari 0, yang artinya makanan selama 31 hari memberi efek kesehatan positif yang kemudian disimpan pada idx_sehat_pos. 

Pada output terdapat 119 dari 500 yang menghasilkan skor sehat positif pada contoh indeks simulasi ke-0, 1, 2, 12, 14, 20, 21, 30, 31, 39 dll. Artinya, pada output terdapat banyak pola makan yang sehat namun tidak sesuai dengan budget yang telah ditentukan.

In [19]:
# (2) Dari hasil sehat positif, ambil yang sisa uang positif
idx_valid = [i for i in idx_sehat_pos if sisa[i] > 0]
print("Jumlah pola makan sehat + sisa uang positif:", len(idx_valid))
print("Contoh indeks valid:", idx_valid[:10])


Jumlah pola makan sehat + sisa uang positif: 0
Contoh indeks valid: []


Kode diatas digunakan untuk mengambil sisa uang positif, terdapat 119 yang sehat setelah itu akan dicek kembali apakah terdapat sisa uang masih positif yang memenuhi syarat yakni sehat positif dan sisa uang positif. Kemudian hasilnya akan disimpan pada idx_valid. 

Pada output tertulis bahwa tidak ada satupun simulasi yang memenuhi kedua syarat diatas, maka dari itu semua pola makanan yang sehat ternyata lebih mengeluarkan lebih banyak budget dari yang sudah ditentukan yakni 1 juta rupiah. 

In [ ]:
# (3)  cari titik dengan skor skumulatif terbesar (skor) di mana nilai kumulatif kesehatannya positif

if len(idx_sehat_pos) > 0:
    idx_max = idx_sehat_pos[np.argmax(skor[idx_sehat_pos])]
    print("Indeks terbaik:", idx_max)
    print("Skor kumulatif:", skor[idx_max])
    print("Sisa uang:", sisa[idx_max])
    print("Nilai sehat:", sehat[idx_max])
else:
    print("Tidak ada pola makan dengan sehat positif.")



Indeks terbaik: 39
Skor kumulatif: 101.84022237207799
Sisa uang: -7870000.0
Nilai sehat: 3.5999999999999996


Source code diatas digunakan untuk mencari titik dengan skor kumuliatif terbesar yang dimana nilai kumulatifnya menghasilkan kesehatan yang psoitif. Kode diatas mengambil idx_sehat_pos yang kemudian dicari skor kumulatif terbesarnya. 

Pada output, terdapat skor kumulatif dengan indeks terbaik yakni pada indeks ke-39, dengan skor kumulatif yakni 101.840 yang artinya pilihan makanan sangat memuaskan dengan nilai sehat yakni 3.59 yang menandakan positif dan pola makanan menyehatkan. Namun, pada sisa uang terdapat nilai -787000 yang artinya pengeluaran tersebut lebih besar dari budget. Jadi, pola makanan yang paling sehat dari sisi skor ternyata tidak realistik karena mengeluarkan banyak uang lebih dan tidak sesuai bduget.

In [18]:
# (4) Tampikan jadwal makanan sesuai pilihan (resp_pilih)
if len(idx_sehat_pos) > 0:
    pilihan_terbaik = resp_pilih[idx_max]

    print("\nJadwal makan 31 hari (pagi & malam):")
    for hari in range(31):
        pagi = int(pilihan_terbaik[2*hari])
        malam = int(pilihan_terbaik[2*hari+1])
        if pagi >= 0 and malam >= 0:  # valid check
            print(f"Hari {hari+1}: Pagi = {makan.loc[pagi,'makanan']}, Malam = {makan.loc[malam,'makanan']}")



Jadwal makan 31 hari (pagi & malam):
Hari 1: Pagi = roti, Malam = warteg
Hari 2: Pagi = nasipadang, Malam = mi instan 
Hari 3: Pagi = bakso, Malam = mi instan 
Hari 4: Pagi = warteg, Malam = nasipadang
Hari 5: Pagi = cireng, Malam = nasigoreng
Hari 6: Pagi = capcay, Malam = capcay
Hari 7: Pagi = capcay, Malam = mieayam
Hari 8: Pagi = nasigoreng, Malam = mi instan 
Hari 9: Pagi = cireng, Malam = mi instan 
Hari 10: Pagi = roti, Malam = bakso
Hari 11: Pagi = mieayam, Malam = ayamgeprek
Hari 12: Pagi = nasigoreng, Malam = mi instan 
Hari 13: Pagi = ayamgeprek, Malam = seblak
Hari 14: Pagi = ayamgeprek, Malam = gorengan
Hari 15: Pagi = ayamgeprek, Malam = capcay
Hari 16: Pagi = capcay, Malam = capcay
Hari 17: Pagi = capcay, Malam = nasigoreng
Hari 18: Pagi = mieayam, Malam = seblak
Hari 19: Pagi = warteg, Malam = capcay
Hari 20: Pagi = ayamgeprek, Malam = cireng
Hari 21: Pagi = nasigoreng, Malam = mi instan 
Hari 22: Pagi = warteg, Malam = gorengan
Hari 23: Pagi = seblak, Malam = roti
Har

Kode ini digunakan untuk membuat program menyusun jaadwal makanan lengkap selama 31 hari dari hasil terbaik. Jadwal ini berisi makanan acak tapi sesuai dengan hasil simulasi skor kumulatif tertinggi. Pada jadwal ini mencerminkan pola strategi makan paling memuaskan dari sisi skor gabungan natara kenyang, rasa, sehat, dan bosan. Namun, dari analisis sebelumnya terjadi pengeluaran yang melebihi budget jika menyesuaikan dengan jadwal yang sehat dan positif ini. 